In [1]:
import seaborn as sns
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.benchmark as benchmark
import torch._dynamo
from torchinfo import summary

In [2]:
torch._dynamo.config.cache_size_limit = 16

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
device = "cpu"

In [3]:
df = sns.load_dataset('diamonds')
df = df[['carat', 'depth', 'table', 'price', 'x', 'y', 'z', 'cut']]

df['cut'] = (df['cut'] == 'Ideal').astype(int)

X = df.drop('cut', axis=1)
y = df['cut']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long).to(device)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long).to(device)

print(X.shape)
print(y.shape)

(53940, 7)
(53940,)


In [4]:
def train(model, criterion, optimizer):
    print(summary(model, input_data=X_train_tensor))

    torch.set_printoptions(threshold=float('inf'))
    for name, param in model.named_parameters():
        print(name)
        print(param)

    epochs = 10000
    best_test_loss = float('inf')

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        train_outputs = model(X_train_tensor)
        train_loss = criterion(train_outputs, y_train_tensor)
        train_loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            test_outputs = model(X_test_tensor)
            test_loss = criterion(test_outputs, y_test_tensor)
            test_predictions = torch.argmax(test_outputs, dim=1)
            test_accuracy = (test_predictions == y_test_tensor).sum().item() / y_test_tensor.size(0)

        if test_loss.item() < best_test_loss:
            best_test_loss = test_loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss.item():.8f}, Test Loss: {test_loss.item():.8f}, Test Accuracy: {test_accuracy:.4f}")

    print()
    print(f"Lowest Test Loss: {best_test_loss:.8f}")

In [ ]:
class DiamondNN(nn.Module):
    def __init__(self, n, num_layers=8):
        super().__init__()
        self.bias = torch.nn.Parameter(torch.zeros(n))

        self.initial = nn.Linear(n * 2, n)

        self.linears = nn.ModuleList()
        for i in range(num_layers):
            self.linears.append(nn.Linear(n * 2, n))

            with torch.no_grad():
                nn.init.zeros_(self.linears[i].weight) # type: ignore
                nn.init.zeros_(self.linears[i].bias)   # type: ignore

        self.final = nn.Linear(n, 1)

        with torch.no_grad():
            nn.init.zeros_(self.final.weight)
            nn.init.zeros_(self.final.bias)

    def forward(self, og_x):
        x = og_x + self.bias

        pos = torch.clamp(x, max=0)
        neg = torch.clamp(x, min=0)
        
        x = self.initial(torch.cat([pos, neg], dim=-1)) + og_x

        for linear in self.linears:
            pos = torch.clamp(x, max=0)
            neg = torch.clamp(x, min=0)
        
            x = linear(torch.cat([pos, neg], dim=-1)) + x

        x = self.final(x)
        return torch.cat([x, -x], dim=-1)

In [12]:
model = torch.compile(DiamondNN(n=X.shape[1]).to(device))
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [13]:
train(model, criterion, optimizer)

Layer (type:depth-idx)                   Output Shape              Param #
OptimizedModule                          [43152, 2]                --
├─DiamondNN: 1-1                         [43152, 2]                7
│    └─Linear: 2-1                       [43152, 7]                105
│    └─ModuleList: 2-2                   --                        --
│    │    └─Linear: 3-1                  [43152, 7]                105
│    │    └─Linear: 3-2                  [43152, 7]                105
│    │    └─Linear: 3-3                  [43152, 7]                105
│    │    └─Linear: 3-4                  [43152, 7]                105
│    │    └─Linear: 3-5                  [43152, 7]                105
│    │    └─Linear: 3-6                  [43152, 7]                105
│    │    └─Linear: 3-7                  [43152, 7]                105
│    │    └─Linear: 3-8                  [43152, 7]                105
│    │    └─Linear: 3-9                  [43152, 7]                105
│    │

In [15]:
torch.set_printoptions(sci_mode=False, precision=10)
for name, param in model.named_parameters():
    print(name)
    print(param)

_orig_mod.bias
Parameter containing:
tensor([-0.0022468383, -0.0477701724,  0.2056664228, -0.1188804060,
        -0.1140687913, -0.1146715134, -0.1765338033], requires_grad=True)
_orig_mod.initial.weight
Parameter containing:
tensor([[     0.2290294617,     -0.0246294681,     -0.1985379606,
             -0.0214819089,     -0.0837015510,     -0.1149658859,
             -0.1799329221,      0.1238445714,      0.1765839458,
              0.1025984660,     -0.0721498579,     -0.4084374607,
              0.1397644281,     -0.0915731713],
        [    -0.2013792098,      0.2705000043,     -0.0619450584,
             -0.0010697439,      0.1264246255,     -0.0753916577,
              0.0350726023,      0.2746153176,      0.1813337803,
              0.1478407979,      0.0907372311,     -0.0272063129,
             -0.0468269289,      0.0051075025],
        [     0.1520957500,     -0.1099474877,     -0.0790599361,
             -0.0870177448,     -0.0533139594,      0.1308937371,
              0.02

In [19]:
model = torch.compile(DiamondNN(n=X.shape[1]).to(device))
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [20]:
train(model, criterion, optimizer)

W0517 11:48:06.384000 60065 torch/_dynamo/convert_frame.py:906] [5/65] torch._dynamo hit config.cache_size_limit (16)
W0517 11:48:06.384000 60065 torch/_dynamo/convert_frame.py:906] [5/65]    function: 'hook' (/Users/nhatt/Downloads/All_Mac/Code/ML_experiments/.venv/lib/python3.11/site-packages/torchinfo/torchinfo.py:592)
W0517 11:48:06.384000 60065 torch/_dynamo/convert_frame.py:906] [5/65]    last reason: 5/23: Cache line invalidated because L['module'] got deallocated
W0517 11:48:06.384000 60065 torch/_dynamo/convert_frame.py:906] [5/65] To log all recompilation reasons, use TORCH_LOGS="recompiles".
W0517 11:48:06.384000 60065 torch/_dynamo/convert_frame.py:906] [5/65] To diagnose recompilation issues, see https://pytorch.org/docs/main/torch.compiler_troubleshooting.html.


Layer (type:depth-idx)                   Output Shape              Param #
OptimizedModule                          [43152, 2]                --
├─DiamondNN: 1-1                         [43152, 2]                7
│    └─Linear: 2-1                       [43152, 7]                105
│    └─ModuleList: 2-2                   --                        --
│    │    └─Linear: 3-1                  [43152, 7]                105
│    │    └─Linear: 3-2                  [43152, 7]                105
│    │    └─Linear: 3-3                  [43152, 7]                105
│    │    └─Linear: 3-4                  [43152, 7]                105
│    │    └─Linear: 3-5                  [43152, 7]                105
│    │    └─Linear: 3-6                  [43152, 7]                105
│    │    └─Linear: 3-7                  [43152, 7]                105
│    │    └─Linear: 3-8                  [43152, 7]                105
│    └─Linear: 2-3                       [43152, 1]                8
Total pa